# Part 4 · RAG, user interface and web analytics

This notebook measures the search function the web app calls (`search.py`), compares the baseline and
improved RAG summaries (`rag.py`), and analyses the usage data the app collects (`web/`) (`analytics.py`).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import display

from fashion_search.search import METHODS, Searcher

# The five test queries of Part 2, plus three that AND matching cannot answer.
QUERIES = {
    3: "women printed kurta cotton",
    4: "men polo tshirt regular fit",
    5: "slim fit jogger men",
    6: "puma men tshirt",
    7: "women denim jacket",
    "typo": "blue jeens",
    "brand not sold": "nike running shoes men",
    "no product has all": "men leather jacket puma",
}

start = time.perf_counter()
searcher = Searcher()
print(f"catalogue and index loaded in {time.perf_counter() - start:.1f} s")

catalogue and index loaded in 2.1 s


## 1. Search function

`Searcher.search(query, method, page)` is the general search function the web app calls. It keeps AND
matching from Parts 2 and 3, ranks with any of the four Part 3 methods (the custom score by default), and
adds what a real search box needs: relaxed matching when no product has every word, a cache of full
rankings for the next pages and the RAG summary, and a Word2Vec model saved to disk.

### 1.1 Results and relaxed matching

In [2]:
rows = []
for qid, query in QUERIES.items():
    found = searcher.search(query)
    rows.append({
        "query": query,
        "results": found.total,
        "relaxed": found.relaxed,
        "terms matched": found.matched_terms if found.relaxed else len(set(found.terms)),
        "of terms": len(set(found.terms)),
        "dropped": ", ".join(found.dropped_terms),
        "top result": found.results[0].title if found.results else None,
    })
pd.DataFrame(rows).set_index("query")

,results,relaxed,terms matched,of terms,dropped,top result
query,,,,,,
women printed kurta cotton,40,False,4,4,,Women Polka Print Cotton Blend A-line Kurta (...
men polo tshirt regular fit,673,False,5,5,,Solid Men Polo Neck Blue T-Shirt
slim fit jogger men,33,False,4,4,,Jogger Fit Men Grey Jeans
puma men tshirt,165,False,3,3,,Striped Men Round Neck Blue T-Shirt
women denim jacket,82,False,3,3,,Full Sleeve Solid Women Denim Jacket
blue jeens,7697,True,1,2,,Slim Men Green Jeans
nike running shoes men,37,True,3,4,nike,Running Shoes For Men (Multicolor)
men leather jacket puma,79,True,3,4,,Full Sleeve Solid Men Leather Jacket


### 1.2 Speed

In [3]:
def timed(query, method, repeat=5):
    """Median time of a search with an empty cache, in milliseconds."""
    times = []
    for _ in range(repeat):
        searcher._ranked.cache_clear()
        times.append(searcher.search(query, method).elapsed_ms)
    return sorted(times)[len(times) // 2]


searcher.search("kurta", "word2vec")  # loads (or trains and saves) the Word2Vec model once
speed = pd.DataFrame({
    label: {query: timed(query, method) for query in QUERIES.values()}
    for method, label in METHODS.items()
}).round(1)
speed.loc["median"] = speed.median().round(1)
display(speed)

found = searcher.search(QUERIES[4], "custom")
print(f"next page from the cache: {searcher.search(QUERIES[4], 'custom', page=2).elapsed_ms:.2f} ms")

,Best match (custom score),BM25,TF-IDF,Word2Vec
women printed kurta cotton,3.9,1.7,1.8,1.8
men polo tshirt regular fit,7.0,2.8,3.0,3.8
slim fit jogger men,3.3,1.6,1.7,1.8
puma men tshirt,3.7,1.6,1.6,1.8
women denim jacket,3.2,1.4,1.5,1.4
blue jeens,38.5,9.9,10.9,25.9
nike running shoes men,4.0,2.5,2.4,2.8
men leather jacket puma,4.3,2.5,2.5,2.8
median,4.0,2.1,2.1,2.3


next page from the cache: 1.16 ms


## 2. RAG

`rag.py` sends the query and the retrieved products to an LLM through the Groq API (set `GROQ_API_KEY` in
`.env`). Each way of building the prompt is a variant in `rag.VARIANTS`.

### 2.1 Baseline

The course template's prompt, which only sees each product's pid and title.

In [4]:
from fashion_search import rag

found = searcher.search(QUERIES[7])
print(rag.baseline_prompt(found.query, found.results)[:1200])


        You are an expert product advisor helping users choose the best option from retrieved e-commerce products.

        ## Instructions:
        1. Identify the single best product that matches the user's request.
        2. Present the recommendation clearly in this format:
        - Best Product: [Product PID] [Product Name]
        - Why: [Explain in plain language why this product is the best fit, referring to specific attributes like price, features, quality, or fit to user’s needs.]
        3. If there is another product that could also work, mention it briefly as an alternative.
        4. If no product is a good fit, return ONLY this exact phrase:
        "There are no good products that fit the request based on the retrieved results."

        ## Retrieved Products:
        - PID: JCKFWC5YW4Z3YYZX, Title: Full Sleeve Solid Women Denim Jacket
- PID: JCKFWC5YGQENYT6B, Title: Full Sleeve Solid Women Denim Jacket
- PID: JCKFWC5ZKPYQ6ZGH, Title: Full Sleeve Solid Women Denim J

TODO(RAG): the baseline's weaknesses, with examples.

### 2.2 Improvements

TODO(RAG): two or three improvements as new entries in `rag.VARIANTS`.

In [5]:
# TODO(RAG): show the improved prompt(s) for the same query.

### 2.3 Baseline against improved

TODO(RAG): compare both on the same queries, including one where no product fits.

In [6]:
# Needs GROQ_API_KEY.
# for query in [...]:
#     found = searcher.search(query)
#     for variant in rag.VARIANTS:
#         print(variant, rag.summarize(query, found.results, variant).text)

## 3. Web analytics

The web app logs requests, queries and clicks in `analytics.AnalyticsStore` (in memory).

### 3.1 Data collection

Already collected by the app: one row per page request (session cookie, path, status, user agent, IP,
referrer), one per results page (query text, terms, ranking method, page, number of results, relaxed or not,
time) and one per product opened (with the query and rank it was clicked from).

TODO(analytics): user context, dwell time and, optionally, sessions.

In [7]:
from fashion_search.analytics import AnalyticsStore

# What the store holds after one search and one click (the web app makes these calls).
store = AnalyticsStore()
found = searcher.search(QUERIES[7])
query_id = store.log_query("demo-session", found.query, found.terms, found.method, found.page,
                           found.total, found.relaxed, found.elapsed_ms)
store.log_click("demo-session", found.results[2].pid, query_id, rank=3)
for name in ["queries", "clicks"]:
    print(name)
    display(store.table(name))

queries


,id,session_id,timestamp,text,terms,method,page,results,relaxed,elapsed_ms
0,1,demo-session,2026-11-21 18:31:40.178308,women denim jacket,"[women, denim, jacket]",custom,1,82,False,1.30525


clicks


,id,session_id,timestamp,query_id,pid,rank
0,1,demo-session,2026-11-21 18:31:40.178340,1,JCKFWC5ZKPYQ6ZGH,3


### 3.2 Data model

TODO(analytics): the star schema.

### 3.3 Dashboard

TODO(analytics): indicators and graphs for the `/dashboard` page.

In [8]:
# TODO(analytics): compute the indicators and draw the dashboard graphs.

## 4. Conclusion

TODO: summarise what the web app, the RAG improvements and the analytics show.